# 07 — GPU Runtime & Dual-Coder 5-Stage Live Benchmark (`Agent 07`)

- **Active Run ID**: `run_20261111_002`
- **Owner**: Agent 07 (`src/runtime/`, `src/model_runtime.py`)
- **Canonical Outputs**: `runs/run_20261111_002/gpu_runtime_config.json` & `runs/run_20261111_002/gpu_runtime_benchmark.json`

## Mission & Architectural Invariants
1. **Runtime Hardware Detection**: Detect the physical accelerator and available VRAM at runtime (never assume hardware from documentation).
2. **Model-Family Diversity & Exact Repo Provenance**:
   - **Coder A**: DeepSeek reasoning-oriented 32B quantized model (`deepseek-ai/DeepSeek-R1-Distill-Qwen-32B` derivative).
   - **Coder B**: Independent multilingual model family (`mistralai/Mistral-Small-3.1-24B-Instruct-2503` quantized preferred; `google/gemma-3-27b-it` quantized alternative). Qwen-derived substitutes for Coder B are strictly blocked.
3. **Real 5-Stage vLLM Benchmark & Empirically Derived Memory Budgets**:
   - Stage 1: Coder A alone
   - Stage 2: Coder B alone
   - Stage 3: Both localhost vLLM servers resident
   - Stage 4: One simultaneous structured request to each
   - Stage 5: Bounded concurrent requests & crash/failure recovery isolation


In [1]:
import importlib.util
import os
import subprocess
import sys
from pathlib import Path

HFG_BRANCH = os.environ.get('HFG_BRANCH', 'main')
HFG_COMMIT = os.environ.get('HFG_COMMIT', '').strip()
HFG_GIT_PULL = os.environ.get('HFG_GIT_PULL', '0') == '1'

def _find_repo_root():
    candidates = [Path.cwd(), *Path.cwd().parents, Path('/content/HFG'), Path('/content')]
    for p in candidates:
        if (p / 'src').is_dir() and ((p / 'pyproject.toml').is_file() or (p / 'data').is_dir()):
            return p.resolve()
    return None

REPO_ROOT = _find_repo_root()
if REPO_ROOT is None and Path('/content').is_dir():
    target = Path('/content/HFG')
    print(f'Repository not found locally; cloning branch {HFG_BRANCH} into {target}...')
    try:
        subprocess.run(['git', 'clone', '--branch', HFG_BRANCH, 'https://github.com/necantis/HFG.git', str(target)], check=True)
        if HFG_COMMIT:
            subprocess.run(['git', '-C', str(target), 'checkout', HFG_COMMIT], check=True)
        REPO_ROOT = _find_repo_root()
    except subprocess.CalledProcessError as exc:
        raise RuntimeError(
            'Automatic git clone failed. If the repo is private, clone it manually or mount Google Drive and cd into HFG.'
        ) from exc
elif REPO_ROOT is not None and (HFG_GIT_PULL or HFG_COMMIT):
    subprocess.run(['git', '-C', str(REPO_ROOT), 'fetch', '--all', '--prune'], check=True)
    if HFG_COMMIT:
        subprocess.run(['git', '-C', str(REPO_ROOT), 'checkout', HFG_COMMIT], check=True)
    elif HFG_GIT_PULL:
        subprocess.run(['git', '-C', str(REPO_ROOT), 'checkout', HFG_BRANCH], check=True)
        subprocess.run(['git', '-C', str(REPO_ROOT), 'pull', '--ff-only', 'origin', HFG_BRANCH], check=True)

if REPO_ROOT is None:
    raise RuntimeError(f'Could not locate HFG repo root from {Path.cwd()}. Clone/open inside HFG first.')

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
os.chdir(REPO_ROOT)
print(f'HFG repo root: {REPO_ROOT}')

_REQUIRED_MODULES = {
    'vllm': 'vllm>=0.7.0',
    'huggingface_hub': 'huggingface_hub>=0.25.0',
    'pydantic': 'pydantic>=2.0.0',
    'pandas': 'pandas>=2.0.0',
    'pyarrow': 'pyarrow>=14.0.0',
    'yaml': 'pyyaml>=6.0.0',
}
_missing_pkgs = [pkg for mod, pkg in _REQUIRED_MODULES.items() if importlib.util.find_spec(mod) is None]
if _missing_pkgs and (Path('/content').is_dir() or os.environ.get('HFG_AUTO_INSTALL') == '1') and os.environ.get('HFG_SKIP_PIP_INSTALL') != '1':
    print(f'Installing missing Colab runtime dependencies: {_missing_pkgs}')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *_missing_pkgs], check=True)
else:
    print('Dependency check:', {m: (importlib.util.find_spec(m) is not None) for m in _REQUIRED_MODULES})



HFG repo root: /content/HFG
Dependency check: {'vllm': True, 'huggingface_hub': True, 'pydantic': True, 'pandas': True, 'pyarrow': True, 'yaml': True}


In [2]:
import json
import shutil
from src.runtime import (
    GPURuntimeConfig,
    detect_hardware_profile,
    resolve_dual_coder_specs,
    validate_model_family_diversity,
    build_and_save_runtime_config,
)

if shutil.which('nvidia-smi'):
    subprocess.run(['nvidia-smi'], check=False)

hw = detect_hardware_profile()
print('Detected Hardware Profile:')
print(json.dumps(hw.to_dict(), indent=2))

if not hw.gpu_available and os.environ.get('HFG_ALLOW_CPU_DEMO') != '1':
    raise RuntimeError(
        'No physical NVIDIA GPU detected. Switch Google Colab Runtime -> Change runtime type -> GPU before running 07_gpu_runtime.ipynb.'
    )


Detected Hardware Profile:
{
  "measured_at_utc": "2026-11-13T18:52:47.518308+00:00",
  "hostname": "7d4213ec7641",
  "os_platform": "Linux 6.6.159+ (x86_64)",
  "cpu_model": "AMD EPYC 9B45",
  "cpu_logical_cores": 48,
  "system_ram_total_mb": 181130.01,
  "system_ram_available_mb": 165898.1,
  "gpu_available": true,
  "gpu_count": 1,
  "gpu_device_name": "NVIDIA RTX PRO 6000 Blackwell Server Edition",
  "gpu_total_vram_mb": 97887.0,
  "gpu_free_vram_mb": 77744.0,
  "gpu_used_vram_mb": 19507.0,
  "cuda_version": "13.0",
  "driver_version": "580.82.07",
  "compute_capability": "12.0",
  "detection_method": "nvidia-smi"
}


In [3]:
# Resolve exact Hugging Face model repositories, pinned commit SHAs, and verify model-family diversity
cfg_probe = GPURuntimeConfig(run_id='run_20261111_002')
resolution = resolve_dual_coder_specs(cfg_probe, verify_remote=True, verify_tokenizer=False)
validate_model_family_diversity(cfg_probe.coder_a_spec, cfg_probe.coder_b_spec)
print('Verified Coder A:', cfg_probe.coder_a_spec.model_repo, '@', cfg_probe.coder_a_spec.model_revision)
print('Verified Coder B:', cfg_probe.coder_b_spec.model_repo, '@', cfg_probe.coder_b_spec.model_revision)
print(json.dumps({'audit_a': resolution['audit_a'], 'audit_b': resolution['audit_b']}, indent=2))


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


Verified Coder A: Valdemardi/DeepSeek-R1-Distill-Qwen-32B-AWQ @ 1de6a3f7b151f6ea0f6d42acb3566e094eb8a264
Verified Coder B: RedHatAI/Mistral-Small-3.1-24B-Instruct-2503-quantized.w4a16 @ dc0a9d9df93998ec3163cabca261715c0bbee4f6
{
  "audit_a": [
    {
      "model_repo": "Valdemardi/DeepSeek-R1-Distill-Qwen-32B-AWQ",
      "quantization": "AWQ-INT4",
      "resolved_revision": "1de6a3f7b151f6ea0f6d42acb3566e094eb8a264",
      "status": "VERIFIED",
      "reason": "Remote Hugging Face model metadata and access verified."
    }
  ],
  "audit_b": [
    {
      "model_repo": "RedHatAI/Mistral-Small-3.1-24B-Instruct-2503-quantized.w4a16",
      "quantization": "W4A16-INT4",
      "resolved_revision": "dc0a9d9df93998ec3163cabca261715c0bbee4f6",
      "status": "VERIFIED",
      "reason": "Remote Hugging Face model metadata and access verified."
    }
  ]
}


In [7]:
# Execute the 5-stage live GPU benchmark and derive memory budgets from measurements
cfg, bench_report, cfg_path = build_and_save_runtime_config(
    base_dir=REPO_ROOT,
    run_id='run_20261111_002',
    resolve_remote_models=True,
    verify_tokenizer=False,
    startup_timeout_sec=900.0,
)
print(f'Saved runtime config: {cfg_path}')
print('live_gpu_benchmark_status =', cfg.live_gpu_benchmark_status)
print('execution_mode =', cfg.execution_mode.value)
print('\n=== 5-Stage Benchmark Summary ===')
print(json.dumps(bench_report['stages'], indent=2))
print('\n=== Derived Memory Budget ===')
print(json.dumps(bench_report['derived_memory_budget'], indent=2))


Saved runtime config: /content/HFG/runs/run_20261111_002/gpu_runtime_config.json
live_gpu_benchmark_status = EXECUTED_LIVE_GPU
execution_mode = SEQUENTIAL_FALLBACK

=== 5-Stage Benchmark Summary ===
{
  "1_coder_a_alone": {
    "coder_id": "coder_a",
    "model_repo": "Valdemardi/DeepSeek-R1-Distill-Qwen-32B-AWQ",
    "model_revision": "1de6a3f7b151f6ea0f6d42acb3566e094eb8a264",
    "model_family": "deepseek",
    "quantization": "AWQ-INT4",
    "server_backend": "vllm_openai_server",
    "model_load_time_sec": 108.0429,
    "actual_vram_after_load_mb": 44478.0,
    "kv_cache_allocation_mb": 23592.96,
    "peak_vram_during_inference_mb": 44478.0,
    "estimated_weights_vram_mb": 18500.0,
    "input_throughput_tok_per_sec": 73.83,
    "output_throughput_tok_per_sec": 57.08,
    "schema_valid_output_rate": 1.0,
    "server_health": true
  },
  "2_coder_b_alone": {
    "coder_id": "coder_b",
    "model_repo": "RedHatAI/Mistral-Small-3.1-24B-Instruct-2503-quantized.w4a16",
    "model_revis